### Experiment Vectorization Approach

In [1]:
####imports section####
from IPython.display import Markdown, display
import tqdm as notebook_tqdm
import os
import warnings
import json
from getpass import getpass
from langchain_community.document_loaders import PyPDFLoader
#Imports for RAG Pipeline and Document Store
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.rankers.transformers_similarity import TransformersSimilarityRanker
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
#Imports for Chat Approach
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack.components.generators.utils import print_streaming_chunk
from haystack import Pipeline
import gradio as gr

warnings.filterwarnings("ignore")

### Load Data

In [2]:
loader = PyPDFLoader("./downloads/taxonomy.pdf")
pages = loader.load_and_split()
print("Size of DOC: \n", len(pages))

Size of DOC: 
 205


### Documentation Store
-> Will store the content and the respective embeddings

In [3]:
#size of documents
range_pages = range(0,len(pages))

#Define a DocStore 
document_store = InMemoryDocumentStore()

#Convert to Haystack Docs
docs = [Document(content=pages[i].page_content, meta=pages[i].metadata) for i in range_pages]

#Embedd documents
doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2", normalize_embeddings=True)
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(docs)

print("Embedding size: ", len(docs_with_embeddings['documents'][0].embedding))

#Write documents -> Ids, Content and embeddings
document_store.write_documents(docs_with_embeddings["documents"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embedding size:  384


205

### Build a RAG Pipeline
- To define a RAG Pipeline we need:
    - A Text Embedder, for embedding user queries, these embeddings will be use to compare to the embedding documents;
    - A Retriever, task to to retrieve the documents that match the user query;
    - Template Prompt, it provides the context;
    - Generator, LLM Model;

In [10]:
template = """
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""
#Define pipeline components
pipe_comp = [
    ('text_embedder', SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2", normalize_embeddings =True)),
    ('retriever', InMemoryEmbeddingRetriever(document_store=document_store, scale_score = True)),
    # ('ranker', TransformersSimilarityRanker(model='cross-encoder/ms-marco-MiniLM-L-6-v2', top_k=25)),
    ('prompt_builder', PromptBuilder(template=template)),
    ('llm', OpenAIGenerator(model="gpt-4o", generation_kwargs = {'temperature': 0.1}))
]

#Define pipeline
basic_rag_pipeline = Pipeline()

#1. Add components
for comp in pipe_comp:
    basic_rag_pipeline.add_component(*comp)


# #2. Connect components
# #LOGIC: Output -> Input
basic_rag_pipeline.connect('text_embedder.embedding', "retriever.query_embedding")
basic_rag_pipeline.connect('retriever', 'prompt_builder.documents')
basic_rag_pipeline.connect('prompt_builder', 'llm')


# #With Ranker
# basic_rag_pipeline.connect('text_embedder', "retriever")
# basic_rag_pipeline.connect('retriever', "ranker")
# basic_rag_pipeline.connect('ranker', 'prompt_builder')
# basic_rag_pipeline.connect('prompt_builder', 'llm')


🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - llm: OpenAIGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> llm.prompt (str)

### Ask Phase

In [11]:
question = "What are the conditions for natural gas be considrered a sustainable activity?"

query = {
    'text_embedder': {
        'text': question
    },

    'prompt_builder': {
        'question': question
    }
}


response = basic_rag_pipeline.run(query)
Markdown(response['llm']['replies'][0])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The provided context does not explicitly mention the conditions under which natural gas can be considered a sustainable activity. However, based on the general principles and criteria outlined for other activities, we can infer some potential conditions that might apply to natural gas to be considered sustainable. These conditions would likely include:

1. **Climate Change Mitigation**:
   - The activity should have direct greenhouse gas (GHG) emissions lower than a specified threshold (e.g., 270 gCO₂e/kWh for certain activities).
   - The activity should develop a strategy to account for and reduce GHG emissions arising from services upstream and downstream of the value chain, including intermediate products, raw materials, transport, maintenance, operations, and end-of-life waste management.

2. **Climate Change Adaptation**:
   - The activity should comply with criteria set out in relevant appendices (e.g., Appendix A) to ensure it is resilient to the impacts of climate change.

3. **Sustainable Use and Protection of Water and Marine Resources**:
   - The activity should comply with criteria set out in relevant appendices (e.g., Appendix B) to ensure it does not harm water and marine resources.

4. **Pollution Prevention and Control**:
   - The activity should comply with criteria set out in relevant appendices (e.g., Appendix C) to ensure it minimizes pollution to air, water, and land.

5. **Protection and Restoration of Biodiversity and Ecosystems**:
   - The activity should comply with criteria set out in relevant appendices (e.g., Appendix D) to ensure it does not harm biodiversity and ecosystems.

6. **Transition to a Circular Economy**:
   - The activity should contribute to the circular economy by ensuring efficient use of resources, minimizing waste, and promoting recycling and reuse.

7. **Life-Cycle Assessment**:
   - The activity should conduct a life-cycle assessment to ensure that it does not increase GHG emissions compared to equivalent primary raw materials. This assessment should be verified by an independent third party.

8. **Compliance with Best Available Techniques (BAT)**:
   - The activity should implement relevant techniques for pollution prevention and control and meet associated emission limits as set out in BAT conclusions.

These inferred conditions align with the general principles of sustainability and the "Do No Significant Harm" (DNSH) criteria outlined for other activities in the provided context. However, for specific and detailed conditions applicable to natural gas, one would need to refer to the exact regulatory documents or guidelines that address natural gas activities directly.

### Chat Approach

In [12]:
#Convert HayStack Pipeline into a Tool
def rag_pipeline_func(query: str):
    result = basic_rag_pipeline.run(
        {
            'text_embedder': {'text': query},
            'prompt_builder': {'question': query}
        }
    )

    return {"reply": result['llm']["replies"][0]}


#Add to tools
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'rag_pipeline_func',
            'description': 'Get information about the documents provided',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {
                        'type': 'string',
                        'description': "The query to use in the search. Infer this from the user's message. It should be a question or a statement",
                    }
                },
                'required': ["query"],
            }
        },
    }
]


messages = [
    ChatMessage.from_system("Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."),
    ChatMessage.from_user("Can you provide description of how activities are considered sustainable?")

]

chat_generator = OpenAIChatGenerator(model="gpt-3.5-turbo", streaming_callback=print_streaming_chunk, generation_kwargs={'temperature': 0.1})
response = chat_generator.run(messages=messages, generation_kwargs={"tools": tools})

In [13]:
#Parse function calling information
function_call = json.loads(response["replies"][0].content)[0]
function_name = function_call["function"]["name"]
function_args = json.loads(function_call["function"]["arguments"])
print("Function Name:", function_name)
print("Function Arguments:", function_args)
# Find the correspoding function and call it with the given arguments
available_functions = {"rag_pipeline_func": rag_pipeline_func}
function_to_call = available_functions[function_name]
function_response = function_to_call(**function_args)
print("Function Response:", function_response)

Function Name: rag_pipeline_func
Function Arguments: {'query': 'How are activities considered sustainable?'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Function Response: {'reply': 'Activities are considered sustainable if they meet specific criteria and comply with various environmental objectives and regulations. Based on the provided context, the sustainability of an activity is determined by its adherence to the following key criteria:\n\n1. **Climate Change Mitigation**:\n   - The activity should have low direct greenhouse gas (GHG) emissions, specifically lower than 270 gCO₂e/kWh if it involves on-site generation of heat/cool or co-generation.\n   - It should develop strategies to account for and reduce GHG emissions throughout the value chain, including intermediate products, raw materials, transport, maintenance, operations, and end-of-life waste management.\n\n2. **Climate Change Adaptation**:\n   - The activity must comply with criteria set out in Appendix A, which involves performing a robust climate risk and vulnerability assessment and implementing adaptation solutions to reduce identified physical climate risks.\n\n3. **

In [14]:
function_message = ChatMessage.from_function(content=json.dumps(function_response), name=function_name)
messages.append(function_message)
response = chat_generator.run(messages=messages, generation_kwargs={"tools": tools})

Activities are considered sustainable if they meet specific criteria and comply with various environmental objectives and regulations. Based on the provided context, the sustainability of an activity is determined by its adherence to the following key criteria:

1. **Climate Change Mitigation**:
   - The activity should have low direct greenhouse gas (GHG) emissions, specifically lower than 270 gCO₂e/kWh if it involves on-site generation of heat/cool or co-generation.
   - It should develop strategies to account for and reduce GHG emissions throughout the value chain, including intermediate products, raw materials, transport, maintenance, operations, and end-of-life waste management.

2. **Climate Change Adaptation**:
   - The activity must comply with criteria set out in Appendix A, which involves performing a robust climate risk and vulnerability assessment and implementing adaptation solutions to reduce identified physical climate risks.

3. **Sustainable Use and Protection of Water

### Chat APP Sim

In [16]:
#CHAT APP
chat_generator = OpenAIChatGenerator(model="gpt-4o", generation_kwargs={'temperature': 0.1})
response = None
messages = [
    ChatMessage.from_system(
        "Present as much relevant information as possible, be descriptive."
    )
]

def chatbot_with_fc(message, history):
    messages.append(ChatMessage.from_user(message))
    response = chat_generator.run(messages=messages, generation_kwargs={"tools": tools})

    while True:
        # if OpenAI response is a tool call
        if response and response["replies"][0].meta["finish_reason"] == "tool_calls":
            function_calls = json.loads(response["replies"][0].content)
            print(response["replies"][0])
            for function_call in function_calls:
                ## Parse function calling information
                function_name = function_call["function"]["name"]
                function_args = json.loads(function_call["function"]["arguments"])

                ## Find the correspoding function and call it with the given arguments
                function_to_call = available_functions[function_name]
                function_response = function_to_call(**function_args)

                ## Append function response to the messages list using `ChatMessage.from_function`
                messages.append(ChatMessage.from_function(content=json.dumps(function_response), name=function_name))
                response = chat_generator.run(messages=messages, generation_kwargs={"tools": tools})

        # Regular Conversation
        else:
            messages.append(response["replies"][0])
            break
    return response["replies"][0].content


demo = gr.ChatInterface(
    fn=chatbot_with_fc,
    examples=[
        "Describe what is the purpose of this document.",
        "What are the targets for cement emissions?",
        "Please describe the low carbon electricity sources in the document.",
    ],
    title="Ask me about the taxonomy!",
)

demo.launch()

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


ChatMessage(content='[{"id": "call_IHkfhreWoUYyhIadZyJwBpBx", "function": {"arguments": "{\\"query\\":\\"What is the purpose of this document?\\"}", "name": "rag_pipeline_func"}, "type": "function"}]', role=<ChatRole.ASSISTANT: 'assistant'>, name=None, meta={'model': 'gpt-4o-2024-05-13', 'index': 0, 'finish_reason': 'tool_calls', 'usage': {'completion_tokens': 23, 'prompt_tokens': 90, 'total_tokens': 113}})


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChatMessage(content='[{"id": "call_XV45TYasSIfEubybnGAMrdlP", "function": {"arguments": "{\\"query\\":\\"Describe what is the purpose of this document.\\"}", "name": "rag_pipeline_func"}, "type": "function"}]', role=<ChatRole.ASSISTANT: 'assistant'>, name=None, meta={'model': 'gpt-4o-2024-05-13', 'index': 0, 'finish_reason': 'tool_calls', 'usage': {'completion_tokens': 24, 'prompt_tokens': 483, 'total_tokens': 507}})


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChatMessage(content='[{"id": "call_Wpy0mMPsdDi9B0F8KXIGCZGc", "function": {"arguments": "{\\"query\\":\\"Describe the low carbon electricity sources mentioned in the document.\\"}", "name": "rag_pipeline_func"}, "type": "function"}]', role=<ChatRole.ASSISTANT: 'assistant'>, name=None, meta={'model': 'gpt-4o-2024-05-13', 'index': 0, 'finish_reason': 'tool_calls', 'usage': {'completion_tokens': 23, 'prompt_tokens': 976, 'total_tokens': 999}})


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChatMessage(content='[{"id": "call_eg7bABsPS7DZwHcWDJNIsDGt", "function": {"arguments": "{\\"query\\":\\"What does the document say about electric vehicles?\\"}", "name": "rag_pipeline_func"}, "type": "function"}]', role=<ChatRole.ASSISTANT: 'assistant'>, name=None, meta={'model': 'gpt-4o-2024-05-13', 'index': 0, 'finish_reason': 'tool_calls', 'usage': {'completion_tokens': 21, 'prompt_tokens': 1585, 'total_tokens': 1606}})


Batches:   0%|          | 0/1 [00:00<?, ?it/s]